# NB5a variants — development variants of the LightGBM bank (validation units only)

D34. The first NB5a run (bank of guide §5.2, forecasting the index level) barely beat persistence and lost to CLIPER on the validation units. Four variants of the forecasting interface are compared on the same units, origins, issue rows, prior and scores, with the setting rule 1 chose for the level bank on 20 Nov 2026 (num_leaves 15, learning rate 0.03, min_data_in_leaf 200), pinned here so this record stays reproducible:
- **level**: forecast k(t + h) (guide §5.2, the first NB5a run);
- **change**: forecast k(t + h) − k(t), then add k(t) back (persistence anchor);
- **level, pooled**: one model per quantile for all horizons, with h as a feature (guide §5.2 ablation);
- **change, pooled**: both.

Development data only; every variant counts as a LightGBM development round (D34). The baselines come from NB5a (they do not depend on LightGBM).

In [1]:
import os, sys, time, json, hashlib
from pathlib import Path
import numpy as np, pandas as pd, yaml
ROOT = Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == 'notebooks' else ROOT
sys.path.insert(0, str(ROOT / 'src'))
os.environ['PYTHONPATH'] = str(ROOT / 'src') + os.pathsep + os.environ.get('PYTHONPATH', '')
os.environ.setdefault('LOKY_MAX_CPU_COUNT', str(os.cpu_count()))
from joblib import Parallel, delayed
from pvcold import config as C, audit as A, fleet as F, forecast as FC
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
N_JOBS = os.cpu_count(); t0 = time.time()
nb5 = json.loads((C.RESULTS / 'nb5a_dev' / 'summary.json').read_text())   # source of the baseline forecasts
LGB_PARAMS = {'num_leaves': 15, 'learning_rate': 0.03, 'min_data_in_leaf': 200}   # pinned: rule 1 for the level bank (D34)
PRIOR = yaml.safe_load((C.RESULTS / 'nb3_index' / 'frozen_config_part1.yaml').read_text())['prior_variant']
ALT = json.loads((C.RESULTS / 'nb1_audit' / 'summary.json').read_text())['campus_altitude_m']
p = F.load_panel(); del p['targets15_kW']
E = p['events'].set_index('event'); U = p['units']; geo15 = p['geometry15']
VAL = E.index[E.set == 'validation'].tolist()
idx = {k: pd.read_parquet(C.RESULTS / 'nb3_index' / f'{k}.parquet').set_index('time') for k in ['k_cens', 'sigma', 'c', 'n']}
gains3 = pd.read_csv(C.RESULTS / 'nb3_index' / 'gains.csv')
base = pd.read_parquet(C.RESULTS / 'nb5a_dev' / 'val_forecasts.parquet').query("model != 'LGB'")   # baselines only
print('LightGBM setting (pinned):', LGB_PARAMS, '| baseline rows:', len(base), sorted(base.model.unique()))

LightGBM setting (pinned): {'num_leaves': 15, 'learning_rate': 0.03, 'min_data_in_leaf': 200} | baseline rows: 1103892 ['A0', 'A1', 'A5', 'A7', 'CLIPER', 'PeEn']


In [2]:
VARIANTS = [('level', False), ('change', False), ('level', True), ('change', True)]
tasks = []
for v in VAL:
    s = E.loc[v, 'start']; S = FC.event_series(v, idx, geo15)
    for d in FC.ORIGIN_DAYS:
        o = s + pd.Timedelta(days=d)
        for tg, pl in VARIANTS:
            tasks.append((v, S, o, min(o + pd.Timedelta(days=30), s + pd.Timedelta(days=90)), tg, pl))
t1 = time.time()
res = Parallel(n_jobs=N_JOBS)(delayed(FC.variant_task)(v, S, o, e, LGB_PARAMS, tg, pl) for v, S, o, e, tg, pl in tasks)
var = pd.concat([r[2] for r in res], ignore_index=True)
fc = pd.concat([base, var.astype({c: 'float32' for c in var.columns if c.startswith('q')})], ignore_index=True)
MODELS = sorted(fc.model.unique())
print('%d tasks in %.0f s; models: %s' % (len(tasks), time.time() - t1, MODELS))

120 tasks in 631 s; models: ['A0', 'A1', 'A5', 'A7', 'CLIPER', 'LGB_change', 'LGB_change_pooled', 'LGB_level', 'LGB_level_pooled', 'PeEn']


In [3]:
QC = [f'q{t:.1f}' for t in FC.TAUS]
lv, gl = FC.member_levels()
Tcache, srows = {}, []
for v in VAL:
    assert E.loc[v, 'set'] == 'validation'
    a = F.event_arrays(v, p)
    g = gains3[gains3.event == v].set_index('donor')['gain'].reindex(a['dn'])
    pr, _, _, _ = F.event_priors(a, g, U, ALT, Tcache)
    B = np.asarray(pr['B_' + PRIOR]); g_m = FC.gain_members(pr[PRIOR], B / np.median(B), gl)
    f = fc[fc.event == v].copy()
    f['target'] = f['t'] + pd.to_timedelta(15 * f['h'], unit='min')
    f['y'] = p['y15'][E.loc[v, 'units']].reindex(f['target']).to_numpy()
    f['gt'] = geo15['Gt'].reindex(f['target']).to_numpy()
    good = f[QC].notna().all(axis=1) & np.isfinite(f['y'])
    cnt = f[good].groupby(['h', 't'])['model'].nunique()
    keep = cnt[cnt == len(MODELS)].index
    f = f.set_index(['h', 't']).loc[lambda x: x.index.isin(keep)].reset_index()
    for (m, h), x in f.groupby(['model', 'h']):
        Y = FC.power_members(x[QC].to_numpy(dtype=float), x['gt'].to_numpy(), g_m, lv)
        srows.append({'event': v, 'model': m, 'h': h, **FC.scores(Y, x['y'].to_numpy())})
sc = pd.DataFrame(srows)
MET = ['nRMSE', 'CRPS', 'PICP80', 'nMBE']
med = sc.groupby(['model', 'h'])[MET].median().reset_index()
for met in MET:
    print(f'median {met}, full window:'); print(med.pivot(index='model', columns='h', values=met).round(4).to_string())
w = sc.pivot_table(index=['event', 'h'], columns='model', values='nRMSE')
wins = pd.DataFrame({m: pd.Series({ref: int(((w[m] < w[ref]).groupby('h').sum()).loc[C.PRIMARY_HORIZON]) for ref in ['A1', 'CLIPER', 'A5', 'LGB_level']})
                     for m in MODELS if m.startswith('LGB')}).T
print('units (of 10) where each LightGBM variant beats the reference at 1 h:'); print(wins.to_string())

median nRMSE, full window:
h                      1       2       4       8       16      24
model                                                            
A0                 0.2342  0.2370  0.2422  0.2523  0.2583  0.2313
A1                 0.1338  0.1664  0.1962  0.2352  0.2670  0.2392
A5                 0.1533  0.1702  0.1909  0.2215  0.2443  0.2250
A7                 0.0879  0.0885  0.0909  0.0933  0.0945  0.0831
CLIPER             0.1318  0.1608  0.1857  0.2127  0.2336  0.2106
LGB_change         0.1293  0.1562  0.1815  0.2109  0.2364  0.2180
LGB_change_pooled  0.1385  0.1614  0.1873  0.2199  0.2436  0.2172
LGB_level          0.1347  0.1615  0.1913  0.2323  0.2522  0.2293
LGB_level_pooled   0.1746  0.1863  0.2032  0.2331  0.2520  0.2312
PeEn               0.2383  0.2411  0.2465  0.2569  0.2630  0.2383
median CRPS, full window:
h                      1       2       4       8       16      24
model                                                            
A0                 0.12

In [4]:
summary = {'notebook': 'NB5a_variants', 'environment': A.versions(), 'runtime_s': round(time.time() - t0, 1),
           'baselines_from': {'notebook': 'NB5a_development', 'run_at': nb5['environment']['run_at'], 'models': sorted(base.model.unique())},
           'lgb_params_pinned': LGB_PARAMS, 'variants': [f'{a}{"_pooled" if b else ""}' for a, b in VARIANTS],
           'median_nRMSE': {m: med[med.model == m].set_index('h')['nRMSE'].round(5).to_dict() for m in MODELS},
           'median_CRPS': {m: med[med.model == m].set_index('h')['CRPS'].round(5).to_dict() for m in MODELS},
           'median_PICP80': {m: med[med.model == m].set_index('h')['PICP80'].round(4).to_dict() for m in MODELS},
           'wins_at_1h_of_10': wins.to_dict('index')}
out = A.write_outputs('nb5a_variants', {'variant_scores': sc, 'variant_medians': med, 'variant_wins_1h': wins.rename_axis('model').reset_index()},
                      summary, parquet={'variant_forecasts': var})
print('written to', out, '|', sorted(x.name for x in out.iterdir()))

written to C:\Users\Dell\Desktop\phd_ehtp\pv paper\results\nb5a_variants | ['figures', 'summary.json', 'variant_forecasts.parquet', 'variant_medians.csv', 'variant_scores.csv', 'variant_wins_1h.csv']
